# BSNL Extension — Mixed-Motive Competitor in Indian Telecom

Extends `02_simulation.ipynb` by adding a government-backed firm (BSNL) with objective $U_B = \alpha\pi_B + (1-\alpha)\,CS$.

**Contents**
1. Parameters and helper functions
2. Stage-game equilibrium (fixed point)
3. Comparison: with vs without BSNL
4. Repeated game — collusion sustainability
5. Comparative statics on α
6. Government subsidy requirement
7. Dynamic simulation — BSNL entry shock

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import minimize_scalar, brentq

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

# ── Linear demand ────────────────────────────────────────
a, b = 100.0, 1.0          # Q(p) = max(a - b*p, 0)

# ── Costs ───────────────────────────────────────────────
c_inc  = 20.0              # incumbent marginal cost
c_bsnl = 26.0              # BSNL MC — 30% above incumbents (legacy overhead)

# ── Market-share model ──────────────────────────────────
s0      = 0.10             # BSNL captive base share (rural / govt subscribers)
gamma   = 0.002            # extra share per ₹1 price advantage over incumbents
s_max   = 0.35             # hard cap on BSNL share (quality/coverage inferiority)

# ── BSNL objective ──────────────────────────────────────
alpha   = 0.50             # profit weight (1-alpha goes to CS)

# ── Industry structure ──────────────────────────────────
n       = 3                # private incumbents: Airtel, Jio, Vi

# ── Government subsidy cap ──────────────────────────────
L_bar   = 100.0            # max per-period loss before govt must intervene

print('Parameters loaded.')
print(f'  a={a}, b={b}')
print(f'  c_inc={c_inc}, c_bsnl={c_bsnl}, alpha={alpha}, gamma={gamma}, s0={s0}')

## 1. Helper Functions

**Market share**: $s_B = \min(s_0 + \gamma\cdot\max(p_\text{inc}-p_B,0),\; s_\text{max})$

**Quantities**: $q_B = s_B\cdot(a - b\,p_B)$, $\quad q_i = \frac{1-s_B}{n}\cdot(a - b\,p_\text{inc})$

**Consumer surplus**: $CS = (a - p_\text{eff})^2/(2b)$ where $p_\text{eff} = s_B p_B + (1-s_B)p_\text{inc}$

**BSNL utility**: $U_B = \alpha\pi_B + (1-\alpha)\,CS$

In [ ]:
def share_bsnl(p_b, p_inc):
    return min(s0 + gamma * max(p_inc - p_b, 0.0), s_max)

def quantities(p_b, p_inc):
    sb  = share_bsnl(p_b, p_inc)
    q_b = sb * max(a - b * p_b, 0.0)
    q_i = (1.0 - sb) * max(a - b * p_inc, 0.0) / n
    return q_b, q_i, sb

def pi_bsnl(p_b, p_inc):
    q_b, _, _ = quantities(p_b, p_inc)
    return (p_b - c_bsnl) * q_b

def pi_inc(p_b, p_inc):
    _, q_i, _ = quantities(p_b, p_inc)
    return (p_inc - c_inc) * q_i

def consumer_surplus(p_b, p_inc):
    sb    = share_bsnl(p_b, p_inc)
    p_eff = sb * p_b + (1.0 - sb) * p_inc
    return max(a - p_eff, 0.0)**2 / (2.0 * b)

def bsnl_utility(p_b, p_inc):
    return alpha * pi_bsnl(p_b, p_inc) + (1.0 - alpha) * consumer_surplus(p_b, p_inc)

print('Helper functions defined.')

## 2. Best-Response Functions and Stage-Game Equilibrium

BSNL solves $\max_{p_B} U_B$ subject to $\pi_B \geq -\bar{L}$ (budget constraint).

Incumbents solve $\max_{p_\text{inc}} \pi_i$ taking $p_B$ as given (BSNL moves first).

Equilibrium is the fixed point of the two best-response maps.

In [ ]:
def bsnl_br(p_inc, alpha_val=None):
    """BSNL welfare-optimal price given incumbent price."""
    a_use = alpha_val if alpha_val is not None else alpha
    def neg_utility(p_b):
        pi = pi_bsnl(p_b, p_inc)
        cs = consumer_surplus(p_b, p_inc)
        return -(a_use * pi + (1.0 - a_use) * cs)
    res   = minimize_scalar(neg_utility, bounds=(10.0, p_inc + 5.0), method='bounded')
    p_opt = res.x
    # Budget constraint: if loss > L_bar, find floor price
    if pi_bsnl(p_opt, p_inc) < -L_bar:
        try:
            p_opt = brentq(lambda p: pi_bsnl(p, p_inc) + L_bar, c_bsnl * 0.5, p_inc)
        except ValueError:
            p_opt = c_bsnl
    return p_opt

def incumbent_br(p_b):
    """Incumbents' jointly optimal (collusive) price given BSNL price."""
    res = minimize_scalar(
        lambda p: -pi_inc(p_b, p),
        bounds=(c_inc + 1e-4, 150.0),
        method='bounded'
    )
    return res.x

def find_equilibrium(p_inc_init=55.0, max_iter=300, tol=1e-7):
    p_inc = p_inc_init
    for _ in range(max_iter):
        p_b       = bsnl_br(p_inc)
        p_inc_new = incumbent_br(p_b)
        if abs(p_inc_new - p_inc) < tol:
            return p_b, p_inc_new
        p_inc = 0.6 * p_inc + 0.4 * p_inc_new   # damped update
    return p_b, p_inc

# ── Solve ────────────────────────────────────────────────
p_b_eq, p_inc_eq = find_equilibrium()

# ── No-BSNL benchmark (monopoly on linear demand: p^m = (a + c_inc) / 2) ──
p_m        = (a + c_inc) / (2 * b)
pi_m_each  = (p_m - c_inc) * max(a - b * p_m, 0.0) / n
cs_no_bsnl = max(a - p_m, 0.0)**2 / (2.0 * b)

# ── Equilibrium outcomes with BSNL ──────────────────────
sb_eq   = share_bsnl(p_b_eq, p_inc_eq)
pi_b_eq = pi_bsnl(p_b_eq, p_inc_eq)
pi_i_eq = pi_inc(p_b_eq, p_inc_eq)
cs_eq   = consumer_surplus(p_b_eq, p_inc_eq)

print('=== Stage-game equilibrium ===')
print(f'  No BSNL  : p_inc={p_m:.2f},  π_each={pi_m_each:.2f},  CS={cs_no_bsnl:.2f}')
print(f'  With BSNL: p_inc={p_inc_eq:.2f}, p_BSNL={p_b_eq:.2f}')
print(f'             π_inc/firm={pi_i_eq:.2f},  π_BSNL={pi_b_eq:.2f}')
print(f'             BSNL share={sb_eq:.3f},  CS={cs_eq:.2f}')
print(f'  BSNL profitable: {pi_b_eq > 0}')
print(f'  CS gain vs no-BSNL: +{cs_eq - cs_no_bsnl:.2f} ({(cs_eq/cs_no_bsnl - 1)*100:.1f}%)')
print(f'  Incumbent price drop: {p_m - p_inc_eq:.2f}')

In [ ]:
# ── Best-response curves ────────────────────────────────
p_inc_grid = np.linspace(c_inc + 1, 80, 80)
bsnl_brs   = [bsnl_br(p) for p in p_inc_grid]

p_b_grid   = np.linspace(10, 60, 80)
inc_brs    = [incumbent_br(p) for p in p_b_grid]

fig, ax = plt.subplots()
ax.plot(p_inc_grid, bsnl_brs,  lw=2, color='C1', label='BSNL BR: p_B*(p_inc)')
# Incumbent BR: maps p_b → p_inc; plot as (p_inc output, p_b input)
ax.plot(inc_brs, p_b_grid,    lw=2, color='C0', label='Incumbent BR: p_inc*(p_B)')
ax.plot(p_inc_eq, p_b_eq, 'r*', ms=15, zorder=5,
        label=f'Equilibrium  p_inc={p_inc_eq:.1f}, p_B={p_b_eq:.1f}')
ax.set_xlabel('Incumbent price  p_inc')
ax.set_ylabel('BSNL price  p_B')
ax.set_title('Best-response curves — stage game with BSNL')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Comparison: With vs Without BSNL

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1 — Prices
ax = axes[0]
ax.bar(['No BSNL\n(p_inc)', 'With BSNL\n(p_inc)', 'BSNL\nprice', 'MC\n(c_inc)'],
       [p_m, p_inc_eq, p_b_eq, c_inc],
       color=['C0', 'C0', 'C1', 'gray'], alpha=0.8)
ax.set_ylabel('Price (₹ stylised)')
ax.set_title('Prices')

# Panel 2 — Profits per firm
ax = axes[1]
ax.bar(['Incumbent\n(no BSNL)', 'Incumbent\n(with BSNL)', 'BSNL\nprofit'],
       [pi_m_each, pi_i_eq, pi_b_eq],
       color=['C0', 'C0', 'C1'], alpha=0.8)
ax.set_ylabel('Profit per firm (₹ stylised)')
ax.set_title('Per-firm Profits')

# Panel 3 — Consumer Surplus
ax = axes[2]
ax.bar(['No BSNL', 'With BSNL'],
       [cs_no_bsnl, cs_eq],
       color=['C0', 'C2'], alpha=0.8)
ax.set_ylabel('Consumer Surplus')
ax.set_title(f'Consumer Surplus  (+{(cs_eq/cs_no_bsnl-1)*100:.1f}%)')

plt.suptitle('Stage-game outcomes: BSNL effect  (α=0.5, γ=0.002, c_B=1.3·c_inc, linear demand)')
plt.tight_layout()
plt.show()

# Summary table
df = pd.DataFrame({
    'Metric':      ['Incumbent price', 'BSNL price', 'π incumbent/firm', 'π BSNL', 'Consumer Surplus', 'BSNL market share'],
    'No BSNL':     [f'{p_m:.2f}', '—', f'{pi_m_each:.2f}', '—', f'{cs_no_bsnl:.2f}', '—'],
    'With BSNL':   [f'{p_inc_eq:.2f}', f'{p_b_eq:.2f}', f'{pi_i_eq:.2f}', f'{pi_b_eq:.2f}', f'{cs_eq:.2f}', f'{sb_eq:.3f}']
})
print(df.to_string(index=False))

## 4. Repeated Game — Collusion Sustainability with BSNL

Deviation: undercutting incumbent captures all $(1-s_B^*)$ incumbent market share (BSNL's captive base is unaffected).

$$\pi^\text{dev} \approx n\cdot\pi^\text{coll}, \quad \pi^\text{nash}=0$$

This gives $\delta^* = (n-1)/n$ — **same formula as without BSNL**.

BSNL lowers the *level* of collusive profits but does not alter the *sustainability condition*.

In [ ]:
ns = np.arange(2, 9)
delta_star = (ns - 1) / ns

fig, ax = plt.subplots()
ax.plot(ns, delta_star, 'o-', lw=2, label='δ* = (n−1)/n  (same with or without BSNL)')
ax.axhline(0.95, ls='--', c='gray', alpha=0.7, label='δ≈0.95 (long horizon)')
ax.set_xlabel('Number of private incumbents (n)')
ax.set_ylabel('Critical δ*')
ax.set_title('BSNL does not change collusion sustainability threshold\n(only lowers the collusive profit level)')
for ni, d in zip(ns, delta_star):
    ax.annotate(f'{d:.2f}', (ni, d), textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

pi_dev     = n * pi_i_eq          # deviator gets all incumbent share at collusive price
pi_nash    = 0.0
print(f'Collusive profit/firm  : {pi_i_eq:.2f}')
print(f'Deviation profit       : {pi_dev:.2f}  (≈ n × π_coll)')
print(f'Nash punishment profit : {pi_nash:.2f}')
print(f'Critical δ* (n={n})     : {(n-1)/n:.3f}')
print()
print('BSNL effect on profits — not on δ*:')
print(f'  Without BSNL: collusive profit/firm = {pi_m_each:.2f}')
print(f'  With BSNL:    collusive profit/firm = {pi_i_eq:.2f}  ({(pi_i_eq/pi_m_each-1)*100:.1f}%)')

## 5. Comparative Statics on α

As α varies from 0 (pure welfare) to 1 (pure profit), trace: BSNL price, BSNL profit, incumbent equilibrium price, and consumer surplus.

In [ ]:
alpha_grid = np.linspace(0.05, 0.95, 40)
results = []

for a_val in alpha_grid:
    # Temporarily override alpha inside bsnl_br via parameter
    def _bsnl_br_a(p_inc):
        def neg_u(p_b):
            pi = pi_bsnl(p_b, p_inc)
            cs = consumer_surplus(p_b, p_inc)
            return -(a_val * pi + (1.0 - a_val) * cs)
        res   = minimize_scalar(neg_u, bounds=(10.0, p_inc + 5.0), method='bounded')
        p_opt = res.x
        if pi_bsnl(p_opt, p_inc) < -L_bar:
            try:
                p_opt = brentq(lambda p: pi_bsnl(p, p_inc) + L_bar, c_bsnl * 0.5, p_inc)
            except ValueError:
                p_opt = c_bsnl
        return p_opt

    # Fixed point with this alpha
    p_inc_loc = 55.0
    for _ in range(200):
        pb_loc    = _bsnl_br_a(p_inc_loc)
        pi_new    = incumbent_br(pb_loc)
        if abs(pi_new - p_inc_loc) < 1e-6:
            break
        p_inc_loc = 0.6 * p_inc_loc + 0.4 * pi_new

    results.append({
        'alpha':    a_val,
        'p_b':      pb_loc,
        'p_inc':    p_inc_loc,
        'pi_bsnl':  pi_bsnl(pb_loc, p_inc_loc),
        'pi_inc':   pi_inc(pb_loc, p_inc_loc),
        'cs':       consumer_surplus(pb_loc, p_inc_loc),
    })

df_alpha = pd.DataFrame(results)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0,0].plot(df_alpha['alpha'], df_alpha['p_b'],   lw=2, color='C1', label='BSNL price')
axes[0,0].plot(df_alpha['alpha'], df_alpha['p_inc'],  lw=2, color='C0', label='Incumbent price')
axes[0,0].axhline(c_bsnl, ls=':', c='C1', alpha=0.5, label=f'BSNL MC={c_bsnl}')
axes[0,0].axhline(p_m,    ls=':', c='C0', alpha=0.5, label=f'No-BSNL p^m={p_m}')
axes[0,0].set_xlabel('α'); axes[0,0].set_ylabel('Price')
axes[0,0].set_title('Equilibrium prices vs α'); axes[0,0].legend(fontsize=8)

axes[0,1].plot(df_alpha['alpha'], df_alpha['pi_bsnl'], lw=2, color='C1')
axes[0,1].axhline(0, c='k', lw=0.8, ls='--')
axes[0,1].axhline(-L_bar, c='red', lw=0.8, ls=':', label=f'Loss floor = -{L_bar}')
axes[0,1].set_xlabel('α'); axes[0,1].set_ylabel('BSNL profit')
axes[0,1].set_title('BSNL profit vs α  (positive = self-sustaining)')
axes[0,1].legend(fontsize=8)

axes[1,0].plot(df_alpha['alpha'], df_alpha['pi_inc'],  lw=2, color='C0')
axes[1,0].axhline(pi_m_each, c='C0', ls=':', alpha=0.5, label=f'No-BSNL benchmark={pi_m_each:.1f}')
axes[1,0].set_xlabel('α'); axes[1,0].set_ylabel('Incumbent profit/firm')
axes[1,0].set_title('Incumbent profit vs α'); axes[1,0].legend(fontsize=8)

axes[1,1].plot(df_alpha['alpha'], df_alpha['cs'],  lw=2, color='C2')
axes[1,1].axhline(cs_no_bsnl, c='C2', ls=':', alpha=0.5, label=f'No-BSNL CS={cs_no_bsnl:.1f}')
axes[1,1].set_xlabel('α'); axes[1,1].set_ylabel('Consumer Surplus')
axes[1,1].set_title('Consumer Surplus vs α'); axes[1,1].legend(fontsize=8)

plt.suptitle('Comparative statics: α (BSNL profit weight)')
plt.tight_layout()
plt.show()

## 6. Government Subsidy Requirement

For each α, compute the per-period subsidy the government must provide if BSNL runs a loss.

Subsidy = $\max(0,\; -\pi_B^*)$

In [ ]:
subsidy = np.maximum(0, -df_alpha['pi_bsnl'])
break_even_alpha = df_alpha.loc[df_alpha['pi_bsnl'] >= 0, 'alpha'].min()

fig, ax = plt.subplots()
ax.fill_between(df_alpha['alpha'], subsidy, alpha=0.3, color='C3')
ax.plot(df_alpha['alpha'], subsidy, lw=2, color='C3', label='Required subsidy/period')
ax.axhline(L_bar, ls='--', c='gray', label=f'Budget cap L̄={L_bar}')
if not np.isnan(break_even_alpha):
    ax.axvline(break_even_alpha, ls=':', c='green', label=f'Break-even α≈{break_even_alpha:.2f}')
ax.set_xlabel('α (BSNL profit weight)')
ax.set_ylabel('Per-period government subsidy')
ax.set_title('Government subsidy requirement vs α\nLeft of break-even: BSNL needs recapitalisation')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Break-even α (BSNL self-sustaining): ≈ {break_even_alpha:.2f}')
print(f'At α=0.50 (our base): π_BSNL = {pi_b_eq:.2f}  — subsidy needed = {max(0, -pi_b_eq):.2f}')

## 7. Dynamic Simulation — BSNL Entry Shock

Periods 0–t_entry: 3-firm pure private collusion at $p^m$, no BSNL.

Periods t_entry+: BSNL enters at its equilibrium price; incumbents adjust to constrained optimum.

This shows BSNL as a structural price anchor — different from Jio's transient price war.

In [ ]:
def simulate_bsnl_entry(T=40, t_entry=15):
    p_inc_path, p_b_path, cs_path, pi_inc_path, pi_b_path = [], [], [], [], []

    for t in range(T):
        if t < t_entry:
            # Pure private collusion, no BSNL
            p_inc_path.append(p_m)
            p_b_path.append(np.nan)
            cs_path.append(cs_no_bsnl)
            pi_inc_path.append(pi_m_each)
            pi_b_path.append(np.nan)
        else:
            # BSNL present at equilibrium prices
            p_inc_path.append(p_inc_eq)
            p_b_path.append(p_b_eq)
            cs_path.append(cs_eq)
            pi_inc_path.append(pi_i_eq)
            pi_b_path.append(pi_b_eq)

    return pd.DataFrame({
        'p_inc': p_inc_path, 'p_bsnl': p_b_path,
        'cs':    cs_path,    'pi_inc': pi_inc_path, 'pi_bsnl': pi_b_path
    })

sim = simulate_bsnl_entry(T=40, t_entry=15)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prices
ax = axes[0]
ax.plot(sim['p_inc'],  lw=2, color='C0', label='Incumbent price')
ax.plot(sim['p_bsnl'], lw=2, color='C1', label='BSNL price', ls='--')
ax.axvline(15, c='red', ls=':', alpha=0.7, label='BSNL enters')
ax.axhline(c_bsnl, ls=':', c='C1', alpha=0.4)
ax.set_xlabel('Period'); ax.set_ylabel('Price')
ax.set_title('Price paths'); ax.legend()

# Profits
ax = axes[1]
ax.plot(sim['pi_inc'],  lw=2, color='C0', label='Incumbent π/firm')
ax.plot(sim['pi_bsnl'], lw=2, color='C1', label='BSNL π', ls='--')
ax.axvline(15, c='red', ls=':', alpha=0.7)
ax.set_xlabel('Period'); ax.set_ylabel('Profit')
ax.set_title('Profit paths'); ax.legend()

# Consumer Surplus
ax = axes[2]
ax.plot(sim['cs'], lw=2, color='C2', label='Consumer Surplus')
ax.axvline(15, c='red', ls=':', alpha=0.7, label='BSNL enters')
ax.set_xlabel('Period'); ax.set_ylabel('CS')
ax.set_title('Consumer Surplus'); ax.legend()

plt.suptitle('BSNL entry shock — permanent structural effect (not a price war)')
plt.tight_layout()
plt.show()

## Summary

| Outcome | No BSNL | With BSNL (α=0.5) | Change |
|---|---|---|---|
| Incumbent collusive price | p^m = 60 | p_inc* < p^m | ↓ (moderate) |
| BSNL price | — | p_B* > c_B | Marginally profitable |
| Incumbent profit/firm | π^m/n | π_i* < π^m/n | ↓ |
| Consumer Surplus | (a−p^m)²/2b | Higher | ↑ |
| Critical δ* | (n−1)/n | (n−1)/n | **Unchanged** |

**Key insight**: BSNL disciplines the *level* of collusive prices through share-stealing, without altering *whether* collusion is sustainable. The welfare gain comes not from disrupting collusion but from a permanent competitive fringe that forces incumbents to price below the unconstrained monopoly optimum.

Contrast with Jio: Jio triggered a full Bertrand collapse (prices → MC); BSNL produces a softer, permanent price moderation. The two mechanisms are complementary — Jio reset the market, BSNL prevents full re-collusion to p^m.